In [ ]:
%%writefile s6e10_survival.py
#!/usr/bin/env python3
"""
S6E10 SURVIVAL v2 (Kaggle Playground Series S6E10, airline passenger satisfaction, metric ROC AUC)
==============================================================================================
Survival-model family only. Loading, encoding, row features, route block and the original-data feature are copied
unchanged from s6e10_lite.py (same helper names). Everything new is below the line "SURVIVAL DESIGN".

RESULT OF v1 ON REAL DATA (2x T4, 30 min, measured by the user)
  Search (150k rows, 3 folds): plain binary XGBoost reference 0.95903; best survival trial 0.95910 (discrete-time, time
  reversed, non-events censored at the 0.54 time-quantile), i.e. the best survival configs behave like classifiers.
  Per family, best search AUC: discrete-time 0.9591, AFT 0.9582, Cox 0.9570; time axis Flight Distance was clearly worst
  (0.93-0.94); arrival/departure delay and arr-dep were best; reversed time and censoring at a quantile beat the textbook
  formulation (own-time censoring, forward time, 0.950-0.954). GBSA 0.9456 and took 970 s of the 1471 s search -> removed.
  Full data 5 folds: dt 0.96051/0.96056, aft 0.95938/0.95944, stacked OOF 0.96063, public LB 0.96011 (OOF-LB gap ~0.0005,
  the usual size). Stage 2 had per-fold AUCs ~0.9607 but a pooled OOF of 0.95786: fold models stop after different numbers
  of rounds, so margins are on different scales (HYPOTHESIS). v2 z-scores every fold's predictions before pooling.

CHANGES IN v2 (all untested on real data except where stated)
  * GBSA removed (and its scikit-survival dependency).
  * Per-fold z-scoring of validation and test predictions (fixes pooled-vs-fold scale mismatch; also in stage 2).
  * Final runs use eta x --final-eta-scale (default 0.5, floor 0.015) and up to 4000 rounds: the search tunes on 150k rows
    with <=800 rounds, the final fit has 4.7x more rows. Your lite notes: xgb d8 lr0.02 0.96110 vs xgb_deep 0.96098.
  * Two Optuna workers (one per GPU, threads, shared sqlite study) -> --workers 2 is the default on a 2-GPU machine.
    If it misbehaves run --workers 1; the study resumes.
  * The AUC early-stopping metric is evaluated every 5th round (it dominated AFT/Cox trial time).
  * cens_q range widened to 0.1-1.0 (the v1 winners sat at 0.54-0.73, near the old lower bound 0.5); search starts from the
    four v1 winners plus two textbook survival controls; 200 trials; top 6 distinct configs get full runs.

IDEAS COVERED (all Optuna-searched; the search decides, not me)
  1 Delay as time, satisfaction as the event: XGBoost survival:cox and survival:aft.
  2 Survival-derived features: out-of-fold risk of each chosen member -> extra columns for an XGBoost classifier ("stage2").
    That is a classifier on survival features; --no-stage2 gives a survival-only stack.
  3 Discrete-time hazard: person-period expansion over bins of the time axis, XGBoost binary:logistic per period,
    score = log cumulative hazard up to the row's own bin.
  4 Flight Distance (route id) as the time axis. All rt_* route features are dropped there, otherwise they decode the time.

SEARCH SPACE  time axis {dep delay, arr delay, arr-dep, flight distance, age} x direction (+/-) x event class
  (dissatisfied or satisfied is the "event") x censoring (censor the other class at its own time, or at the q-quantile of
  time) x family x hyper-parameters; use the original-data feature or not; keep the partner delay column.
  RULE ENFORCED: the time-axis column and everything derived from it are removed from the covariates. Otherwise Cox
  and AFT learn the time instead of the label (degenerate; checked on synthetic data: Cox 0.66 -> 0.54, AFT 0.69 -> 0.58,
  and rt_* kept with time=Flight Distance: Cox 0.74 -> 0.56). A discrete-time model that keeps the time covariate is just a
  classifier; that variant is opt-in via --allow-dt-time.
  TIES: delays have many exact zeros. Cox/AFT times get a random jitter smaller than the smallest gap between distinct
  values; discrete-time bins use un-jittered times.

REQUIRES  xgboost>=2 (custom_metric, device=), optuna, scikit-learn.

RUN (Kaggle, 2x T4)
  %%writefile s6e10_survival.py   <- paste this file        then:   !python s6e10_survival.py
  python s6e10_survival.py --quick                 # minutes: end-to-end check on 60k rows
  python s6e10_survival.py                         # 200 trials (<=60 min) + full 5-fold runs of the best 6 configs
  --trials 120 --top 4 shorter | --workers 1 single GPU | --final-eta-scale 1 same eta as the search | --fresh new study
  The Optuna study lives in <out>/surv_optuna_*.db and resumes; finished full-data members are cached as surv_member_*.npz.
  Output: submission.csv (id, satisfaction = probability), surv_trials.csv, survival_oof_test.npz (per-fold z-scored OOF and
  test scores, higher = more satisfied; same StratifiedKFold(5, seed 42) split as the lite pipeline, so they can be added
  to its stack).

CAVEATS
  * The search score is the best of ~200 trials on a 150k-row subsample: optimistic. Trust the full-data OOF numbers.
  * Stage 2 uses out-of-fold survival scores with the shared fold split (standard stacking, slight leak).
  * Selection compares AUC only. A member can win the search and still add nothing to a stack of your existing models.
  * Two-worker GPU threading and the 0.5 eta scale are untested on the real data.
"""
import argparse
import gc
import glob
import hashlib
import json
import os
import queue
import shutil
import sys
import threading
import time
import warnings

import numpy as np
import pandas as pd
import xgboost as xgb
from scipy.optimize import minimize
from scipy.special import expit
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

try:
    import optuna
except ImportError:
    sys.exit("optuna missing: pip install optuna")

warnings.filterwarnings("ignore")
T0 = time.time()
TARGET = "satisfaction"
SEED = 42
OPT_ROUNDS, OPT_ES = 800, 50


def log(m):
    print(f"[{time.time() - T0:5.0f}s] {m}", flush=True)


STEP = [0]


def step(m):
    STEP[0] += 1
    print(f"\n[{time.time() - T0:5.0f}s] ===== STEP {STEP[0]}: {m} =====", flush=True)


class OutOfTime(Exception):
    pass


def args_():
    p = argparse.ArgumentParser()
    p.add_argument("--data", default="", help="competition folder (auto-detected on Kaggle)")
    p.add_argument("--og", default="", help="original-dataset folder (auto-detected; 'none' to disable)")
    p.add_argument("--folds", type=int, default=5, help="folds for the final full-data runs")
    p.add_argument("--rows", type=int, default=0, help="subsample the training rows (sanity run)")
    p.add_argument("--opt-rows", type=int, default=150000, help="rows used inside the Optuna search")
    p.add_argument("--opt-folds", type=int, default=3)
    p.add_argument("--trials", type=int, default=200, help="max Optuna trials (also stops at --opt-min)")
    p.add_argument("--opt-min", type=float, default=60.0, help="Optuna wall-clock budget in minutes")
    p.add_argument("--top", type=int, default=6, help="distinct best configs that get a full-data run")
    p.add_argument("--max-min", type=float, default=120.0, help="soft budget for the full-data phase (checked between folds)")
    p.add_argument("--xgb-rounds", type=int, default=4000)
    p.add_argument("--final-eta-scale", type=float, default=0.5, help="final full-data fits use eta x this (floor 0.015)")
    p.add_argument("--workers", type=int, default=0, help="parallel Optuna trials (0 = one per GPU, max 2)")
    p.add_argument("--allow-dt-time", action="store_true", help="let the discrete-time model keep the time axis as a covariate (then it is just a classifier)")
    p.add_argument("--no-stage2", action="store_true", help="skip the classifier on survival-derived features")
    p.add_argument("--fresh", action="store_true", help="delete the existing Optuna study first")
    p.add_argument("--quick", action="store_true", help="tiny end-to-end check (forces 3 folds, top 2, <=12 trials, <=60k rows)")
    p.add_argument("--cpu", action="store_true")
    p.add_argument("--out", default="/kaggle/working" if os.path.isdir("/kaggle/working") else ".")
    return p.parse_args()


# ------------------------------------------------------------------------------------------------ data (verbatim from s6e10_lite.py)
def norm(s):
    return str(s).strip().lower()


def find_dirs(a):
    comp = a.data
    if not comp:
        c = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/sample_submission.csv", recursive=True)]
        c = [d for d in c if os.path.exists(d + "/train.csv") and os.path.exists(d + "/test.csv")]
        comp = sorted(c, key=lambda d: "s6e10" not in d)[0]
    og = a.og
    if og != "none" and not og:
        c = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/train.csv", recursive=True)]
        c = [d for d in c if "airline" in d.lower() and "playground" not in d.lower()]
        og = c[0] if c else ""
    return comp, ("" if og == "none" else og)


def load(a):
    comp, ogd = find_dirs(a)
    tr, te = pd.read_csv(comp + "/train.csv"), pd.read_csv(comp + "/test.csv")
    sub = pd.read_csv(comp + "/sample_submission.csv") if os.path.exists(comp + "/sample_submission.csv") else None
    y = tr[TARGET].astype(str).str.lower().isin(["true", "1", "1.0", "satisfied"]).to_numpy().astype(int)
    cols = [c for c in te.columns if norm(c) != "id"]
    idc = [c for c in te.columns if norm(c) == "id"][0]
    og_x, og_y = None, None
    if ogd:
        try:
            parts = [pd.read_csv(f"{ogd}/{f}") for f in ("train.csv", "test.csv") if os.path.exists(f"{ogd}/{f}")]
            og = pd.concat(parts, ignore_index=True)
            og.columns = [c.strip() for c in og.columns]
            m = {norm(c): c for c in og.columns}
            if all(norm(c) in m for c in cols) and "satisfaction" in m:
                og_x = og[[m[norm(c)] for c in cols]].copy()
                og_x.columns = cols
                og_y = og[m["satisfaction"]].astype(str).str.lower().isin(["satisfied", "true", "1"]).to_numpy().astype(int)
                keep = ~og_x.assign(_y=og_y).duplicated().to_numpy()
                og_x, og_y = og_x[keep].reset_index(drop=True), og_y[keep]
        except Exception as e:
            log(f"original data not usable: {e!r}")
    log(f"train={len(tr):,} test={len(te):,} features={len(cols)} positive rate={y.mean():.4f} | "
        f"original data: {'none' if og_x is None else f'{len(og_x):,} rows'}")
    return tr[cols], te[cols], y, te[idc].to_numpy(), idc, sub, og_x, og_y, cols


def encode(tr_x, te_x, og_x):
    """Everything to float32. Strings -> integer codes shared across all frames. Returns frames + metadata."""
    frames = [tr_x, te_x] + ([og_x] if og_x is not None else [])
    ALL = pd.concat([f.reset_index(drop=True) for f in frames], ignore_index=True)
    cat_cols = [c for c in ALL.columns if not pd.api.types.is_numeric_dtype(ALL[c])]   # strings (object or str dtype)
    for c in ALL.columns:
        if c in cat_cols:
            v = ALL[c].astype(str).str.strip().str.lower()
            ALL[c] = pd.Categorical(v, categories=sorted(v.unique())).codes.astype(np.float32)
        else:
            v = pd.to_numeric(ALL[c], errors="coerce")
            ALL[c] = v.fillna(v.median()).astype(np.float32)
    n1, n2 = len(tr_x), len(te_x)
    parts = [ALL.iloc[:n1], ALL.iloc[n1:n1 + n2]] + ([ALL.iloc[n1 + n2:]] if og_x is not None else [])
    parts = [p.reset_index(drop=True) for p in parts]
    ratings = [c for c in ALL.columns if c not in cat_cols and ALL[c].nunique() <= 7 and ALL[c].min() >= 0 and ALL[c].max() <= 5]
    low = {c: norm(c) for c in ALL.columns}
    pick = lambda *keys: [c for c in ALL.columns if any(k in low[c] for k in keys)]
    meta = dict(cat=cat_cols, ratings=ratings, fd=pick("distance")[0], age=[c for c in ALL.columns if low[c] == "age"][0],
                dep=[c for c in pick("delay") if "depart" in low[c]][0], arr=[c for c in pick("delay") if "arriv" in low[c]][0],
                bg=[c for k in ("class", "type of travel", "customer type") for c in pick(k)][:3])
    return parts, meta


# ------------------------------------------------------------------------------------------------ features (verbatim)
def basic(X, meta, pairs=None):
    """Row-level features: raw columns + zero-count + non-zero mean (+ zero-pair flags) + delay difference."""
    F = X.copy()
    R = X[meta["ratings"]].to_numpy()
    Z = R == 0
    F["zero_count"] = Z.sum(1).astype(np.float32)
    nz = (~Z).sum(1)
    F["nonzero_mean"] = np.where(nz > 0, (R * (~Z)).sum(1) / np.maximum(nz, 1), 0).astype(np.float32)
    F["delay_diff"] = X[meta["arr"]] - X[meta["dep"]]
    return F


def route_block(F, meta):
    """Label-free route context: Flight Distance = route id. Computed on train+test together."""
    key = F[meta["fd"]].round().astype(np.int64)
    cols = meta["ratings"] + ["zero_count", "nonzero_mean", meta["age"], meta["dep"], meta["arr"]] + meta["bg"]
    G = F[cols].groupby(key).transform("mean").astype(np.float32)
    G.columns = ["rt_" + c for c in cols]
    G["rt_count"] = key.map(key.value_counts()).astype(np.float32)
    return G


def xgb_params(gpu, depth, eta, seed=SEED):
    p = dict(objective="binary:logistic", eval_metric="auc", eta=eta, max_depth=depth, subsample=0.8, colsample_bytree=0.5,
             min_child_weight=5, reg_lambda=2.0, tree_method="hist", seed=seed, max_bin=256)
    p["device"] = "cuda" if gpu else "cpu"
    if not gpu:
        p["nthread"] = os.cpu_count() or 1
    return p


def og_feature(F_all, F_og, y_og, y_tr, n_tr, meta, gpu):
    """Model trained ONLY on the original data, scored on every competition row (no competition label used)."""
    d = xgb.DMatrix(F_og.to_numpy(np.float32), y_og)
    b = xgb.train(xgb_params(gpu, 6, 0.1), d, 300)
    m = b.predict(xgb.DMatrix(F_all.to_numpy(np.float32)), output_margin=True)
    log(f"original-only xgb: AUC on competition labels {roc_auc_score(y_tr, m[:n_tr]):.5f} (expect ~0.955); margin std {m.std():.2f}")
    key = F_all[meta["fd"]].round().astype(np.int64)
    out = pd.DataFrame({"og_xgb": m.astype(np.float32)})
    out["og_route_mean"] = pd.Series(m).groupby(key.to_numpy()).transform("mean").astype(np.float32)
    return out


# ------------------------------------------------------------------------------------------------ cv / stack (verbatim)
def cv(name, fit, y, folds, n_te, budget_end):
    """fit(i_tr, i_va) -> (validation margins, test margins). Returns OOF and mean test margins."""
    oof, te = np.zeros(len(y)), np.zeros(n_te)
    t = time.time()
    for k, (a, b) in enumerate(folds):
        if time.time() > budget_end:
            raise OutOfTime(f"{name}: out of time before fold {k + 1}")
        pv, pt = fit(a, b)
        oof[b] = pv
        te += pt / len(folds)
        log(f"{name} fold {k + 1}/{len(folds)} AUC {roc_auc_score(y[b], pv):.5f} ({(time.time() - t) / 60:.1f} min in member)")
    log(f"{name}: OOF AUC {roc_auc_score(y, oof):.5f}")
    return oof, te


def fit_xgb_factory(X, Xte, y, gpu, rounds):
    def fit(a, b):
        da, db = xgb.DMatrix(X[a], y[a]), xgb.DMatrix(X[b], y[b])
        bst = xgb.train(xgb_params(gpu, 8, 0.06), da, rounds, evals=[(db, "v")], early_stopping_rounds=100, verbose_eval=False)
        it = (0, bst.best_iteration + 1)
        return bst.predict(db, iteration_range=it, output_margin=True), bst.predict(xgb.DMatrix(Xte), iteration_range=it, output_margin=True)
    return fit


def stack(Z, y):
    """Non-negative logistic stacker on member logits (>= 0 weights stop collinear members cancelling out)."""
    Z = Z.astype(np.float64)

    def f(p):
        s = Z @ p[:-1] + p[-1]
        r = expit(s) - y
        return np.mean(np.logaddexp(0, s) - y * s), np.append(Z.T @ r, r.sum()) / len(y)
    p0 = np.append(np.full(Z.shape[1], 1.0 / Z.shape[1]), 0.0)
    r = minimize(f, p0, jac=True, method="L-BFGS-B", bounds=[(0, None)] * Z.shape[1] + [(None, None)])
    return r.x[:-1], r.x[-1]


# ================================================================================================ SURVIVAL DESIGN
AXES = ["dep", "arr", "dd", "fd", "age"]
_TLS = threading.local()                       # which GPU the current Optuna worker thread uses


def axis_col(meta, axis):
    return {"dep": meta["dep"], "arr": meta["arr"], "dd": "delay_diff", "fd": meta["fd"], "age": meta["age"]}[axis]


def time_axis(F, meta, axis, direction, tf, jitter):
    """Positive time per row (train+test), label-free. Ties get a random jitter smaller than the smallest gap."""
    v = F[axis_col(meta, axis)].to_numpy(np.float64)
    u = v - v.min()
    if direction < 0:
        u = u.max() - u
    u = u + 1.0
    if tf == "sqrt":
        u = np.sqrt(u)
    elif tf == "log":
        u = 1.0 + np.log(u)
    elif tf == "rank":
        r = pd.Series(u).rank(method="average").to_numpy()
        u = 1.0 + 999.0 * (r - 1.0) / max(len(u) - 1, 1)
    if jitter:
        uq = np.unique(u)
        gap = float(np.diff(uq).min()) if len(uq) > 1 else 1.0
        u = u + np.random.RandomState(SEED).uniform(0.0, 0.99 * gap, size=len(u))
    return u


def cov_idx(cols, meta, cfg):
    """Covariates = every feature except the time axis and anything derived from it (see module docstring)."""
    ax, drop = cfg["axis"], set()
    if cfg.get("keep_time_cov"):                 # discrete-time only: keeps the time axis, i.e. a classifier in disguise
        return np.array([i for i, c in enumerate(cols) if cfg["use_og"] or not c.startswith("og_")], dtype=np.int64)
    if ax == "dep":
        drop |= {meta["dep"], "delay_diff"}
    elif ax == "arr":
        drop |= {meta["arr"], "delay_diff"}
    elif ax == "dd":
        drop |= {meta["dep"], meta["arr"], "delay_diff"}
    elif ax == "fd":
        drop |= {meta["fd"], "og_route_mean"} | {c for c in cols if c.startswith("rt_")}
    elif ax == "age":
        drop |= {meta["age"]}
    if ax in ("dep", "arr") and not cfg["keep_partner"]:
        drop |= {meta["dep"], meta["arr"], "delay_diff"}
    if not cfg["use_og"]:
        drop |= {c for c in cols if c.startswith("og_")}
    return np.array([i for i, c in enumerate(cols) if c not in drop], dtype=np.int64)


def design(cfg, ctx):
    """Time vector, event flags and censoring times for a config (cached per time definition)."""
    key = (cfg["axis"], cfg["dir"], cfg["tf"], cfg["family"] == "xgb_dt")
    if key not in ctx["T"]:
        ctx["T"][key] = time_axis(ctx["F"], ctx["meta"], cfg["axis"], cfg["dir"], cfg["tf"], jitter=cfg["family"] != "xgb_dt")
    T, n, y = ctx["T"][key], ctx["n_tr"], ctx["y"]
    ev = (y == 0) if cfg["event"] == "dis" else (y == 1)
    c = T[:n].copy()
    if cfg["cens"] == "q":                      # the non-event class is observed at least up to the q-quantile of time
        c = np.where(ev, c, np.maximum(c, np.quantile(T[:n], cfg["cens_q"])))
    return dict(T=T, ev=ev, c=c, idx=cov_idx(ctx["cols"], ctx["meta"], cfg))


def tparams(cfg, ctx, obj):
    p = dict(objective=obj, eta=cfg["eta"], max_depth=cfg["depth"], subsample=cfg["subsample"], colsample_bytree=cfg["colsample"],
             min_child_weight=cfg["mcw"], reg_lambda=cfg["lam"], tree_method="hist", seed=SEED, max_bin=256)
    gpu = ctx["gpu"] and obj not in ctx["force_cpu"]
    p["device"] = f"cuda:{getattr(_TLS, 'dev', 0)}" if gpu else "cpu"
    if not gpu:
        p["nthread"] = os.cpu_count() or 1
    return p


def train_safe(p, dtr, rounds, dva, es, ctx, metric=None):
    kw = dict(evals=[(dva, "v")], early_stopping_rounds=es, verbose_eval=False)
    if metric is not None:
        p = dict(p, disable_default_eval_metric=1)
        kw.update(custom_metric=metric, maximize=True)
    try:
        return xgb.train(p, dtr, rounds, **kw)
    except xgb.core.XGBoostError as e:
        if not str(p.get("device", "")).startswith("cuda"):
            raise
        log(f"GPU failed for {p['objective']} ({str(e)[:90]}); using CPU for it from now on")
        ctx["force_cpu"].add(p["objective"])
        q = dict(p, device="cpu", nthread=os.cpu_count() or 1)
        return xgb.train(q, dtr, rounds, **kw)


def auc_metric(evb, sign, every=5):
    """Early-stopping metric. AUC is computed every `every`-th round only (stale value in between): it was the main cost of
    AFT/Cox trials. AFT can diverge to inf/nan: such rounds score 0.5 instead of crashing."""
    st = {"n": 0, "last": 0.5}

    def m(pred, _d):
        st["n"] += 1
        if st["n"] % every == 1 or every == 1:
            v = sign * np.asarray(pred, dtype=np.float64)
            st["last"] = roc_auc_score(evb, v) if np.isfinite(v).all() else 0.5
        return "auc_ev", st["last"]
    return m


def expand(bins):
    """Person-period rows: row i appears once per period 0..bins[i]. Returns (source row, period)."""
    cnt = bins + 1
    rep = np.repeat(np.arange(len(bins)), cnt)
    k = np.arange(int(cnt.sum())) - np.repeat(np.cumsum(cnt) - cnt, cnt)
    return rep, k


def dt_risk(bst, it, Xrows, bins):
    """log cumulative hazard over periods 0..own bin (higher = more likely the event)."""
    rep, k = expand(bins)
    h = bst.predict(xgb.DMatrix(np.hstack([Xrows[rep], k[:, None].astype(np.float32)])), iteration_range=it)
    s = np.bincount(rep, weights=-np.log1p(-np.clip(h, 1e-9, 1 - 1e-9)), minlength=len(bins))
    return np.log(np.maximum(s, 1e-12))


def fit_member(cfg, a, b, ctx, with_test, rounds, es):
    """Train on train-row indices a, score train-row indices b (and test). Risk orientation: higher = event."""
    d, fam, X, n = design(cfg, ctx), cfg["family"], ctx["X"], ctx["n_tr"]
    idx, ev, c, T = d["idx"], d["ev"], d["c"], d["T"]
    Xa, Xb = X[np.ix_(a, idx)], X[np.ix_(b, idx)]
    Xt = X[n:][:, idx] if with_test else None
    if fam in ("xgb_cox", "xgb_aft"):
        if fam == "xgb_cox":
            lab = np.where(ev, c, -c).astype(np.float32)          # xgboost: negative label = right-censored
            da, db = xgb.DMatrix(Xa, label=lab[a]), xgb.DMatrix(Xb, label=lab[b])
            p, sign = tparams(cfg, ctx, "survival:cox"), 1.0
        else:
            lo, up = c.astype(np.float32), np.where(ev, c, np.inf).astype(np.float32)
            da, db = xgb.DMatrix(Xa), xgb.DMatrix(Xb)
            for m_, ix in ((da, a), (db, b)):
                m_.set_float_info("label_lower_bound", lo[ix])
                m_.set_float_info("label_upper_bound", up[ix])
            p, sign = tparams(cfg, ctx, "survival:aft"), -1.0       # prediction is a time: short time = high risk
            p.update(aft_loss_distribution=cfg["dist"], aft_loss_distribution_scale=cfg["scale"],
                     base_score=float(np.median(c[a])))         # AFT base_score is in time units (initial margin = log of it)
        bst = train_safe(p, da, rounds, db, es, ctx, auc_metric(ev[b], sign))
        it = (0, bst.best_iteration + 1)
        rb = sign * bst.predict(db, iteration_range=it, output_margin=True)
        rt = sign * bst.predict(xgb.DMatrix(Xt), iteration_range=it, output_margin=True) if with_test else None
        return rb, rt, it[1]
    if fam == "xgb_dt":
        edges = np.unique(np.quantile(T[:n], np.linspace(0, 1, cfg["nb"] + 1)[1:-1]))
        binof = lambda t: np.searchsorted(edges, t, side="left")
        bc = binof(c)                                               # training bin: own time for events, censor time otherwise

        def rows(Xm, ix):
            rep, k = expand(bc[ix])
            return np.hstack([Xm[rep], k[:, None].astype(np.float32)]), ((k == bc[ix][rep]) & ev[ix][rep]).astype(np.float32)
        (Ea, ya), (Eb, yb) = rows(Xa, a), rows(Xb, b)
        p = tparams(cfg, ctx, "binary:logistic")
        p["eval_metric"] = "logloss"
        bst = xgb.train(p, xgb.DMatrix(Ea, label=ya), rounds, evals=[(xgb.DMatrix(Eb, label=yb), "v")], early_stopping_rounds=es, verbose_eval=False)
        it = (0, bst.best_iteration + 1)
        rb = dt_risk(bst, it, Xb, binof(T[:n][b]))
        rt = dt_risk(bst, it, Xt, binof(T[n:])) if with_test else None
        return rb, rt, it[1]
    raise ValueError(fam)


# ------------------------------------------------------------------------------------------------ Optuna
def sample_cfg(t, og_ok, dt_time):
    c = {}
    c["family"] = t.suggest_categorical("family", ["xgb_cox", "xgb_aft", "xgb_dt"])
    c["axis"] = t.suggest_categorical("axis", AXES)
    c["dir"] = t.suggest_categorical("dir", [1, -1])
    c["event"] = t.suggest_categorical("event", ["dis", "sat"])
    c["cens"] = t.suggest_categorical("cens", ["own", "q"])
    c["cens_q"] = t.suggest_float("cens_q", 0.1, 1.0) if c["cens"] == "q" else 1.0
    c["keep_partner"] = t.suggest_categorical("keep_partner", [False, True]) if c["axis"] in ("dep", "arr") else False
    c["use_og"] = t.suggest_categorical("use_og", [True, False]) if og_ok else False
    c["tf"] = t.suggest_categorical("tf", ["raw", "sqrt", "log", "rank"]) if c["family"] == "xgb_aft" else "raw"
    if c["family"] == "xgb_aft":
        c["dist"] = t.suggest_categorical("dist", ["normal", "logistic", "extreme"])
        c["scale"] = t.suggest_float("scale", 0.3, 3.0, log=True)
    if c["family"] == "xgb_dt":
        c["nb"] = t.suggest_int("nb", 3, 10)
        c["keep_time_cov"] = t.suggest_categorical("keep_time_cov", [False, True]) if dt_time else False
    c["depth"] = t.suggest_int("depth", 3, 10)
    c["eta"] = t.suggest_float("eta", 0.04, 0.2, log=True)
    c["mcw"] = t.suggest_float("mcw", 1.0, 40.0, log=True)
    c["subsample"] = t.suggest_float("subsample", 0.5, 1.0)
    c["colsample"] = t.suggest_float("colsample", 0.3, 0.9)
    c["lam"] = t.suggest_float("lam", 0.3, 20.0, log=True)
    return c


def make_objective(a, ctx, ofolds, og_ok, devq):
    def objective(trial):
        cfg = sample_cfg(trial, og_ok, a.allow_dt_time)
        trial.set_user_attr("cfg", cfg)
        dev = devq.get()                                 # one trial per GPU at a time
        _TLS.dev = dev
        try:
            aucs = []
            for k, (tr_i, va_i) in enumerate(ofolds):
                rb, _, _ = fit_member(cfg, tr_i, va_i, ctx, False, OPT_ROUNDS, OPT_ES)
                aucs.append(roc_auc_score(design(cfg, ctx)["ev"][va_i], rb))
                trial.report(float(np.mean(aucs)), k)
                if k < len(ofolds) - 1 and trial.should_prune():
                    raise optuna.TrialPruned()
            return float(np.mean(aucs))
        except optuna.TrialPruned:
            raise
        except Exception as e:
            log(f"trial {trial.number} FAILED: {e!r}")
            raise
        finally:
            devq.put(dev)
            gc.collect()
    return objective


def start_trials():
    """Optuna starts from the v1 winners (partial settings, the rest is sampled) plus two textbook survival controls."""
    return [dict(family="xgb_dt", axis="arr", dir=-1, event="dis", cens="q", cens_q=0.54),
            dict(family="xgb_dt", axis="dep", dir=-1, event="dis", cens="q", cens_q=0.64),
            dict(family="xgb_aft", axis="arr", dir=-1, event="sat", cens="q", cens_q=0.73),
            dict(family="xgb_aft", axis="dep", dir=-1, event="dis", cens="q", cens_q=0.63),
            dict(family="xgb_cox", axis="dep", dir=1, event="dis", cens="own"),
            dict(family="xgb_dt", axis="dep", dir=1, event="dis", cens="own")]


def pick_top(study, k):
    done = sorted([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE and t.value is not None], key=lambda t: -t.value)
    chosen, seen = [], set()
    for t in done:                                   # first pass: one config per (family, time axis, event class)
        key = (t.params["family"], t.params["axis"], t.params["event"])
        if key not in seen and len(chosen) < k:
            chosen.append(t)
            seen.add(key)
    for t in done:                                   # fill up with the next best
        if len(chosen) < k and t not in chosen:
            chosen.append(t)
    return chosen


def cfg_name(c):
    return (f"{c['family']}|{c['axis']}{'-' if c['dir'] < 0 else '+'}|ev={c['event']}|cens={c['cens']}" + (f"{c['cens_q']:.2f}" if c["cens"] == "q" else "")
            + ("|+timecov" if c.get("keep_time_cov") else ""))


def cached(path, fn):
    if os.path.exists(path):
        z = np.load(path)
        log(f"cache hit {os.path.basename(path)}")
        return z["oof"], z["te"]
    oof, te = fn()
    np.savez(path, oof=oof, te=te)
    return oof, te


def zs(v):
    return (v - v.mean()) / (v.std() + 1e-12)


def run_final(cfg, ctx, folds, n_te, end, rounds, es, eta_scale):
    """Full-data CV of one config. Each fold's validation and test predictions are z-scored before pooling/averaging, so fold
    models that stopped after different numbers of rounds (different margin scales) pool correctly."""
    cfg = dict(cfg, eta=max(0.015, cfg["eta"] * eta_scale))
    oof, te = np.zeros(ctx["n_tr"]), np.zeros(n_te)
    ev, aucs, t = design(cfg, ctx)["ev"], [], time.time()
    for k, (tr_i, va_i) in enumerate(folds):
        if time.time() > end:
            raise OutOfTime(f"{cfg_name(cfg)}: out of time before fold {k + 1}")
        rb, rt, it = fit_member(cfg, tr_i, va_i, ctx, True, rounds, es)
        aucs.append(roc_auc_score(ev[va_i], rb))
        oof[va_i] = zs(rb)
        te += zs(rt) / len(folds)
        log(f"  fold {k + 1}/{len(folds)} AUC {aucs[-1]:.5f} ({it} rounds, eta {cfg['eta']:.3f}, {(time.time() - t) / 60:.1f} min in member)")
    log(f"  mean fold AUC {np.mean(aucs):.5f} | pooled OOF AUC {roc_auc_score(ev, oof):.5f}")
    return oof, zs(te)


def cv_z(name, fit, y, folds, n_te, budget_end):
    """Like cv() from the lite script, but z-scores each fold's predictions (see run_final) and logs pooled vs fold-mean AUC."""
    oof, te, aucs, t = np.zeros(len(y)), np.zeros(n_te), [], time.time()
    for k, (a, b) in enumerate(folds):
        if time.time() > budget_end:
            raise OutOfTime(f"{name}: out of time before fold {k + 1}")
        pv, pt = fit(a, b)
        aucs.append(roc_auc_score(y[b], pv))
        oof[b] = zs(pv)
        te += zs(pt) / len(folds)
        log(f"{name} fold {k + 1}/{len(folds)} AUC {aucs[-1]:.5f} ({(time.time() - t) / 60:.1f} min in member)")
    log(f"{name}: mean fold AUC {np.mean(aucs):.5f} | pooled OOF AUC {roc_auc_score(y, oof):.5f}")
    return oof, zs(te)


def reference_auc(ctx, ofolds):
    """Plain binary XGBoost on all covariates with the same search folds: what the survival members must be compared to."""
    X, y = ctx["X"], ctx["y"]
    aucs = []
    for tr_i, va_i in ofolds:
        da, db = xgb.DMatrix(X[tr_i], label=y[tr_i]), xgb.DMatrix(X[va_i], label=y[va_i])
        bst = xgb.train(xgb_params(ctx["gpu"], 6, 0.1), da, OPT_ROUNDS, evals=[(db, "v")], early_stopping_rounds=OPT_ES, verbose_eval=False)
        aucs.append(roc_auc_score(y[va_i], bst.predict(db, iteration_range=(0, bst.best_iteration + 1), output_margin=True)))
    return float(np.mean(aucs))


# ------------------------------------------------------------------------------------------------ main
def n_gpus():
    if shutil.which("nvidia-smi") is None:
        return 0
    try:
        import subprocess
        out = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True, timeout=20).stdout
        return sum(1 for ln in out.splitlines() if ln.startswith("GPU"))
    except Exception:
        return 1


def main():
    a = args_()
    if a.quick:
        a.rows = a.rows or 60000
        a.opt_rows, a.folds, a.trials, a.top = min(a.opt_rows, 20000), 3, min(a.trials, 12), min(a.top, 2)
        a.opt_min, a.xgb_rounds = min(a.opt_min, 8.0), min(a.xgb_rounds, 300)
    ng = 0 if a.cpu else n_gpus()
    gpu = ng > 0
    workers = a.workers or (min(2, ng) if gpu else 1)
    os.makedirs(a.out, exist_ok=True)
    log(f"survival v2 run: gpus={ng} workers={workers} folds={a.folds} rows={a.rows or 'all'} opt_rows={a.opt_rows} trials<={a.trials} opt_min={a.opt_min} top={a.top} out={a.out}")

    step("loading data")
    tr_x, te_x, y, ids, idc, sub, og_x, og_y, cols = load(a)
    if a.rows and a.rows < len(y):
        keep = np.random.RandomState(SEED).choice(len(y), a.rows, replace=False)
        tr_x, y = tr_x.iloc[keep].reset_index(drop=True), y[keep]
        log(f"subsampled to {len(y):,} training rows")
    n_tr, n_te = len(tr_x), len(te_x)

    step("encoding + row features + route context (unchanged from lite)")
    parts, meta = encode(tr_x, te_x, og_x)
    log(f"ratings={len(meta['ratings'])} categorical={meta['cat']} route key='{meta['fd']}' dep='{meta['dep']}' arr='{meta['arr']}'")
    Fall = basic(pd.concat(parts[:2], ignore_index=True), meta)
    blocks = [Fall, route_block(Fall, meta)]
    og_ok = og_x is not None
    if og_ok:
        blocks.append(og_feature(Fall, basic(parts[2], meta), og_y, y, n_tr, meta, gpu))
    Fdf = pd.concat(blocks, axis=1)
    X = Fdf.to_numpy(np.float32)
    log(f"feature matrix {X.shape[0]:,} x {X.shape[1]}")
    ctx = dict(F=Fdf, X=X, cols=list(Fdf.columns), meta=meta, y=y, n_tr=n_tr, gpu=gpu, T={}, force_cpu=set())
    folds = list(StratifiedKFold(a.folds, shuffle=True, random_state=SEED).split(np.zeros(n_tr), y))
    osub = np.random.RandomState(SEED).choice(n_tr, min(a.opt_rows, n_tr), replace=False)
    ofolds = [(osub[i], osub[j]) for i, j in StratifiedKFold(a.opt_folds, shuffle=True, random_state=SEED).split(np.zeros(len(osub)), y[osub])]

    step(f"Optuna search on {len(osub):,} rows x {a.opt_folds} folds, {workers} worker(s)")
    try:
        log(f"reference (plain binary XGBoost, all covariates, same folds): AUC {reference_auc(ctx, ofolds):.5f}")
    except Exception as e:
        log(f"reference run failed: {e!r}")
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    name = f"survv2_{len(osub)}_{a.opt_folds}_{int(og_ok)}_{n_tr}"
    db = os.path.join(a.out, f"surv_optuna_{name}.db")
    if a.fresh and os.path.exists(db):
        os.remove(db)
    kw = dict(seed=SEED, multivariate=True, group=True, n_startup_trials=15)
    try:
        sampler = optuna.samplers.TPESampler(constant_liar=workers > 1, **kw)
    except TypeError:
        sampler = optuna.samplers.TPESampler(**kw)
    storage = optuna.storages.RDBStorage(f"sqlite:///{db}", engine_kwargs={"connect_args": {"timeout": 60}})
    study = optuna.create_study(direction="maximize", study_name=name, storage=storage, load_if_exists=True, sampler=sampler,
                                pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=0))
    if not study.trials:
        for p in start_trials():
            study.enqueue_trial(p, skip_if_exists=True)

    def cb(st, tr):
        v = tr.value if tr.value is not None else float("nan")
        log(f"trial {tr.number:3d} {tr.state.name:8s} AUC {v:.5f} | " + " ".join(f"{k}={tr.params[k]}" for k in ("family", "axis", "dir", "event", "cens") if k in tr.params))

    left = a.trials - len([t for t in study.trials if t.state.is_finished()])
    if left > 0:
        devq = queue.Queue()
        for i in range(workers):
            devq.put(i % max(ng, 1))
        study.optimize(make_objective(a, ctx, ofolds, og_ok, devq), n_trials=left, timeout=a.opt_min * 60, n_jobs=workers,
                       callbacks=[cb], catch=(Exception,))
    study.trials_dataframe().to_csv(os.path.join(a.out, "surv_trials.csv"), index=False)
    top = pick_top(study, a.top)
    if not top:
        sys.exit("no Optuna trial completed: see FAILED lines above")
    log("best trials (search AUC, config, hyper-parameters):")
    for t in top:
        c = t.user_attrs["cfg"]
        log(f"  {t.value:.5f}  {cfg_name(c)}  " + json.dumps({k: (round(v, 4) if isinstance(v, float) else v) for k, v in c.items() if k in ("depth", "eta", "mcw", "subsample", "colsample", "lam", "nb", "dist", "scale", "tf", "use_og", "keep_partner")}))

    step(f"full-data runs of the top {len(top)} configs ({a.folds} folds, eta x {a.final_eta_scale})")
    end = time.time() + a.max_min * 60
    sig = f"v2z_{a.folds}_{n_tr}_{int(og_ok)}_{a.final_eta_scale}_{a.xgb_rounds}"
    mem = {}
    for t in top:
        cfg = t.user_attrs["cfg"]
        path = os.path.join(a.out, "surv_member_" + hashlib.md5((json.dumps(cfg, sort_keys=True) + sig).encode()).hexdigest()[:12] + ".npz")
        log(f"member {cfg_name(cfg)} (search AUC {t.value:.5f})")
        try:
            oof, te = cached(path, lambda: run_final(cfg, ctx, folds, n_te, end, a.xgb_rounds, 150, a.final_eta_scale))
        except OutOfTime as e:
            log(f"DROPPED: {e}")
            continue
        except Exception as e:
            log(f"FAILED: {e!r}")
            continue
        sgn = 1.0 if cfg["event"] == "sat" else -1.0           # orient every member as "higher = more satisfied"
        mem[cfg_name(cfg) + f"#{t.number}"] = (sgn * oof, sgn * te)
        log(f"  pooled OOF AUC vs satisfaction {roc_auc_score(y, sgn * oof):.5f}")
    if not mem:
        sys.exit("no survival member finished: raise --max-min or lower --rows")

    if not a.no_stage2:
        step("stage 2: XGBoost classifier on base features + out-of-fold survival scores")
        So, St = np.column_stack([m[0] for m in mem.values()]), np.column_stack([m[1] for m in mem.values()])
        Xs = np.hstack([X, np.vstack([So, St]).astype(np.float32)])
        path = os.path.join(a.out, "surv_stage2_" + hashlib.md5((",".join(mem) + sig).encode()).hexdigest()[:12] + ".npz")
        try:
            oof, te = cached(path, lambda: cv_z("stage2", fit_xgb_factory(Xs[:n_tr], Xs[n_tr:], y, gpu, a.xgb_rounds), y, folds, n_te, time.time() + 3600))
            mem["stage2_xgb(base+survival scores)"] = (oof, te)
        except Exception as e:
            log(f"stage 2 FAILED: {e!r}")

    step(f"stacking {len(mem)} member(s)")
    names = list(mem)
    Zo, Zt = np.column_stack([mem[n][0] for n in names]), np.column_stack([mem[n][1] for n in names])
    w, b = (np.array([1.0]), 0.0) if len(names) == 1 else stack(Zo, y)
    for i, (n, wi) in enumerate(zip(names, w)):
        log(f"  m{i} weight {wi:.3f}  OOF AUC {roc_auc_score(y, mem[n][0]):.5f}  {n}")
    if len(names) > 1:
        log("Spearman between members (OOF):\n" + pd.DataFrame(Zo, columns=[f"m{i}" for i in range(len(names))]).corr(method="spearman").round(4).to_string())
    log(f"FINAL OOF AUC {roc_auc_score(y, Zo @ w + b):.5f} | simple mean {roc_auc_score(y, Zo.mean(1)):.5f}")

    step("writing submission.csv")
    prob = expit(Zt @ w + b)
    out = pd.DataFrame({(sub.columns[0] if sub is not None else idc): ids, (sub.columns[1] if sub is not None else TARGET): prob})
    if sub is not None:
        assert len(out) == len(sub), "row count differs from sample_submission"
    out.to_csv(os.path.join(a.out, "submission.csv"), index=False)
    np.savez(os.path.join(a.out, "survival_oof_test.npz"), names=np.array(names), oof=Zo, te=Zt, w=w, b=b)
    log(f"wrote {a.out}/submission.csv ({len(out):,} rows, mean prob {prob.mean():.4f} vs train rate {y.mean():.4f}); total {(time.time() - T0) / 60:.1f} min")


if __name__ == "__main__":
    main()


In [ ]:
# first time: add --quick for a minutes-long end-to-end check. Resume after a timeout = rerun the same command.
# --workers 1 if two-GPU threading misbehaves | --trials 120 --top 4 for a shorter run | --no-stage2 = survival-only stack
!pip install -q optuna
!python s6e10_survival.py
